# 02. Model Training — Distracted Driver Detection

**Model**: EfficientNetB0 (Transfer Learning from ImageNet)  
**Dataset**: State Farm Distracted Driver Detection (10 classes)  
**Strategy**: Fine-tune on training split, evaluate on validation split (subject-level, no leakage)

> Set `SAMPLE_MODE = True` để train nhanh với tập nhỏ (kiểm tra pipeline).  
> Set `SAMPLE_MODE = False` để train đầy đủ.

## 0. Config

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# ─── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ─── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR      = '..'
DATA_DIR      = os.path.join(BASE_DIR, 'data', 'raw', 'statefarm')
IMG_DIR       = os.path.join(DATA_DIR, 'imgs', 'train')
PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'processed')
MODEL_DIR     = os.path.join(BASE_DIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# ─── Hyperparameters ───────────────────────────────────────────────────────────
SAMPLE_MODE   = True    # True = dùng tập nhỏ để kiểm tra pipeline nhanh
SAMPLE_PER_CLASS = 50   # Số ảnh/class khi SAMPLE_MODE = True

IMG_SIZE      = 224     # EfficientNetB0 input size
BATCH_SIZE    = 32
EPOCHS        = 20      # Max epochs (early stopping sẽ dừng sớm nếu cần)
LEARNING_RATE = 1e-3
NUM_CLASSES   = 10

CLASS_NAMES = [f'c{i}' for i in range(NUM_CLASSES)]
CLASS_LABELS = {
    'c0': 'Normal driving',
    'c1': 'Texting (right)',
    'c2': 'Phone (right)',
    'c3': 'Texting (left)',
    'c4': 'Phone (left)',
    'c5': 'Radio',
    'c6': 'Drinking',
    'c7': 'Reaching behind',
    'c8': 'Hair & makeup',
    'c9': 'Talking to passenger',
}

print(f'TensorFlow version : {tf.__version__}')
print(f'GPUs available     : {len(tf.config.list_physical_devices("GPU"))}')
print(f'Sample mode        : {SAMPLE_MODE}')
if SAMPLE_MODE:
    print(f'  => {SAMPLE_PER_CLASS} images/class  ({SAMPLE_PER_CLASS * NUM_CLASSES} total)')

## 1. Load Metadata

In [ ]:
train_df = pd.read_csv(os.path.join(PROCESSED_DIR, 'train_metadata.csv'))
val_df   = pd.read_csv(os.path.join(PROCESSED_DIR, 'val_metadata.csv'))

print(f'Full train set : {len(train_df)} images')
print(f'Full val set   : {len(val_df)} images')

# ─── Sample nếu SAMPLE_MODE ────────────────────────────────────────────────────
if SAMPLE_MODE:
    train_df = (
        train_df
        .groupby('classname', group_keys=False)
        .apply(lambda x: x.sample(min(SAMPLE_PER_CLASS, len(x)), random_state=SEED))
        .reset_index(drop=True)
    )
    val_df = (
        val_df
        .groupby('classname', group_keys=False)
        .apply(lambda x: x.sample(min(SAMPLE_PER_CLASS // 5, len(x)), random_state=SEED))
        .reset_index(drop=True)
    )
    print(f'\n[SAMPLE MODE] Sampled train : {len(train_df)} images')
    print(f'[SAMPLE MODE] Sampled val   : {len(val_df)} images')

# Tạo full image path
train_df['filepath'] = train_df.apply(
    lambda r: os.path.join(IMG_DIR, r['classname'], r['img']), axis=1
)
val_df['filepath'] = val_df.apply(
    lambda r: os.path.join(IMG_DIR, r['classname'], r['img']), axis=1
)

# Encode label thành integer
label2idx = {c: i for i, c in enumerate(sorted(CLASS_NAMES))}
train_df['label'] = train_df['classname'].map(label2idx)
val_df['label']   = val_df['classname'].map(label2idx)

print('\nClass distribution (train):')
print(train_df['classname'].value_counts().sort_index())

## 2. Build tf.data Pipeline

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_image(filepath, label):
    """Đọc ảnh từ đĩa, decode và resize."""
    img = tf.io.read_file(filepath)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32)
    # EfficientNet expects pixels in [0, 255] — preprocessing built-in
    return img, label


def augment(img, label):
    """Data augmentation cho training set."""
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, max_delta=0.2)
    img = tf.image.random_contrast(img, lower=0.8, upper=1.2)
    img = tf.image.random_saturation(img, lower=0.8, upper=1.2)
    img = tf.clip_by_value(img, 0.0, 255.0)
    return img, label


def make_dataset(df, training=False):
    filepaths = df['filepath'].values
    labels    = df['label'].values.astype(np.int32)

    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)

    if training:
        ds = ds.shuffle(buffer_size=len(df), seed=SEED)
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)
    return ds


train_ds = make_dataset(train_df, training=True)
val_ds   = make_dataset(val_df,   training=False)

print('Train batches :', len(train_ds))
print('Val batches   :', len(val_ds))

## 3. Visualise Sample Batch

In [ ]:
idx2class = {v: k for k, v in label2idx.items()}

images, labels = next(iter(train_ds))
n_show = min(12, BATCH_SIZE)

plt.figure(figsize=(16, 8))
for i in range(n_show):
    plt.subplot(3, 4, i + 1)
    img = images[i].numpy().astype(np.uint8)
    cls = idx2class[labels[i].numpy()]
    plt.imshow(img)
    plt.title(f'{cls}\n{CLASS_LABELS[cls]}', fontsize=8)
    plt.axis('off')
plt.suptitle('Sample Training Images (after augmentation)', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Build Model (EfficientNetB0 Transfer Learning)

In [ ]:
def build_model(num_classes=NUM_CLASSES, learning_rate=LEARNING_RATE):
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    # EfficientNetB0: include_top=False, pretrained ImageNet weights
    # include_preprocessing=True → accepts raw [0,255] pixels
    base = EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_tensor=inputs,
        include_preprocessing=True,   # built-in normalization
    )
    base.trainable = False  # Freeze base — Phase 1: train classifier head only

    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model


model = build_model()
model.summary(line_length=100)

## 5. Phase 1 — Train Classifier Head (Base Frozen)

In [ ]:
callbacks_phase1 = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1
    ),
]

print('=' * 60)
print('Phase 1: Training classifier head (base frozen)')
print('=' * 60)

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks_phase1,
    verbose=1,
)

## 6. Phase 2 — Fine-tune (Unfreeze Top Layers)

In [ ]:
# Unfreeze top 30 layers của EfficientNetB0 để fine-tune
base_model = model.layers[1]   # EfficientNetB0 là layer index 1
base_model.trainable = True

# Chỉ unfreeze 30 layers cuối
for layer in base_model.layers[:-30]:
    layer.trainable = False

total_trainable = sum(1 for l in model.layers if l.trainable)
print(f'Trainable layers after unfreeze: {total_trainable}')

# Recompile với LR nhỏ hơn
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE / 10),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks_phase2 = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(MODEL_DIR, 'best_model.keras'),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1,
    ),
]

print('=' * 60)
print('Phase 2: Fine-tuning top layers')
print('=' * 60)

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks_phase2,
    verbose=1,
)

## 7. Training Curves

In [ ]:
def plot_history(h1, h2=None):
    acc  = h1.history['accuracy']
    vacc = h1.history['val_accuracy']
    loss = h1.history['loss']
    vloss= h1.history['val_loss']

    if h2:
        acc  += h2.history['accuracy']
        vacc += h2.history['val_accuracy']
        loss += h2.history['loss']
        vloss+= h2.history['val_loss']
        phase2_start = len(h1.history['accuracy'])
    else:
        phase2_start = None

    epochs_range = range(len(acc))

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Accuracy
    axes[0].plot(epochs_range, acc,  label='Train Acc')
    axes[0].plot(epochs_range, vacc, label='Val Acc')
    if phase2_start:
        axes[0].axvline(x=phase2_start - 0.5, color='gray', linestyle='--', label='Fine-tune start')
    axes[0].set_title('Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    axes[0].grid(True)

    # Loss
    axes[1].plot(epochs_range, loss,  label='Train Loss')
    axes[1].plot(epochs_range, vloss, label='Val Loss')
    if phase2_start:
        axes[1].axvline(x=phase2_start - 0.5, color='gray', linestyle='--', label='Fine-tune start')
    axes[1].set_title('Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    axes[1].grid(True)

    plt.suptitle('Training History', fontsize=14)
    plt.tight_layout()
    plt.show()

plot_history(history1, history2)

## 8. Evaluate on Validation Set

In [ ]:
val_loss, val_acc = model.evaluate(val_ds, verbose=0)
print(f'Validation Loss     : {val_loss:.4f}')
print(f'Validation Accuracy : {val_acc:.4f} ({val_acc*100:.2f}%)')

## 9. Confusion Matrix & Classification Report

In [ ]:
# Lấy toàn bộ predictions
y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# ─── Confusion Matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
)
plt.title('Confusion Matrix (Validation Set)', fontsize=14)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

# ─── Classification Report ─────────────────────────────────────────────────────
print('\nClassification Report:')
print(classification_report(
    y_true, y_pred,
    target_names=[CLASS_LABELS[c] for c in CLASS_NAMES]
))

## 10. Save Final Model

In [ ]:
mode_tag = 'sample' if SAMPLE_MODE else 'full'
save_path = os.path.join(MODEL_DIR, f'efficientnetb0_{mode_tag}.keras')
model.save(save_path)
print(f'Model saved to: {save_path}')

# In kết quả tổng kết
print()
print('=' * 50)
print('  SUMMARY')
print('=' * 50)
print(f'  Mode           : {"SAMPLE" if SAMPLE_MODE else "FULL"}')
print(f'  Train images   : {len(train_df)}')
print(f'  Val images     : {len(val_df)}')
print(f'  Val Accuracy   : {val_acc*100:.2f}%')
print(f'  Val Loss       : {val_loss:.4f}')
print(f'  Saved to       : {save_path}')
print('=' * 50)